#**Installation des packages nécessaires**

In [2]:
import time # gestion des délais

import requests # requêtes HTTP

import pandas as pd # manipulation de données

from bs4 import BeautifulSoup # analyse HTML

import numpy as np # calculs numériques

import re # expressions régulières

from datetime import datetime # gestion des dates et heures

import os # interaction avec le système de fichiers

# **1. Collecte de données et web scraping**

In [3]:
# Liste des URLs des pages Wikipédia à scraper
wiki_urls = [
    "https://en.wikipedia.org/wiki/Opinion_polling_for_the_2022_Philippine_presidential_election",
    "https://en.wikipedia.org/wiki/Opinion_polling_for_the_2016_Philippine_presidential_election",
    "https://en.wikipedia.org/wiki/Opinion_polling_for_the_2010_Philippine_presidential_election",
    "https://en.wikipedia.org/wiki/Opinion_polling_for_the_2023_Polish_parliamentary_election",
    "https://en.wikipedia.org/wiki/Opinion_polling_for_the_2019_Polish_parliamentary_election",
    "https://en.wikipedia.org/wiki/Opinion_polling_for_the_2015_Polish_parliamentary_election",
    "https://en.wikipedia.org/wiki/Opinion_polling_for_the_2011_Polish_parliamentary_election",
    "https://en.wikipedia.org/wiki/Opinion_polling_for_the_2007_Polish_parliamentary_election",
    "https://en.wikipedia.org/wiki/Opinion_polling_for_the_2005_Polish_parliamentary_election",
    "https://en.wikipedia.org/wiki/Opinion_polling_for_the_2001_Polish_parliamentary_election"
]

# Dictionnaire pour stocker les résultats {nom élection: [df_tableau1, df_tableau2]}
scraped_data = {}

def scrape_polling_tables(url):
    start_time = time.time()

    # Extraire le nom de l'élection depuis l'URL
    election_name = url.split("/")[-1].replace("_", " ")

    print(f"Scraping : {election_name} ...")

    # Récupération et parsing de la page
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()  # Vérifie si la requête a réussi
    except requests.RequestException as e:
        print(f"Erreur lors de la requête : {e}")
        return None

    soup = BeautifulSoup(response.text, 'html.parser')

    # Trouver tous les tableaux wikitable
    tables = soup.find_all('table', class_='wikitable')

    if len(tables) < 2:
        print(f"Moins de deux tableaux trouvés pour {election_name}.")
        return None

    # Liste pour stocker les DataFrames de cette page
    dataframes = []

    # Boucle pour scraper les deux premiers tableaux
    for i, table in enumerate(tables[:2]):
        rows = table.find_all('tr')

        # Extraire les en-têtes (première ligne)
        headers = [header.get_text(strip=True) for header in rows[0].find_all('th')]

        # Extraction des données
        data = []
        for row in rows[1:]:  # Ignorer l'en-tête
            cols = row.find_all(['td', 'th'])
            row_data = []

            for col in cols:
                colspan = int(col.get("colspan", 1))  # Gérer les colonnes fusionnées
                rowspan = int(col.get("rowspan", 1))  # Gérer les lignes fusionnées
                text = col.get_text(strip=True)


                if col.name == "td":
                   for _ in range(colspan):
                       row_data.append(text)
                else:
                    row_data.append(text)  # Ajouter normalement les <th>


            # Ajuster si le nombre de colonnes diffère
            if len(row_data) != len(headers):
                row_data = row_data[:len(headers)] if len(row_data) > len(headers) else row_data + [''] * (len(headers) - len(row_data))

            data.append(row_data)

        # Création du DataFrame
        df = pd.DataFrame(data, columns=headers)
        dataframes.append(df)

    end_time = time.time()
    print(f"Scraping terminé pour {election_name} en {end_time - start_time:.4f} secondes.\n")

    return dataframes

# Exécuter le scraping sur toutes les pages
for url in wiki_urls:
    result = scrape_polling_tables(url)
    if result:
        election_name = url.split("/")[-1].replace("_", " ")
        scraped_data[election_name] = result

# Affichage des résultats
for election, dfs in scraped_data.items():
    print(f"\n🔹 Élection : {election}")
    for i, df in enumerate(dfs):
        print(f"Tableau {i+1} :")
        print(df.to_string(), "\n")

# Dictionnaire pour stocker les DataFrames
election_tables = {}

for election, dfs in scraped_data.items():
    for i, df in enumerate(dfs):
        table_name = f"{election.replace(' ', '_')}_table_{i+1}"
        election_tables[table_name] = df  # Stocker le DataFrame sous un nom unique
        print(f"✅ Tableau stocké sous le nom : {table_name}")



Scraping : Opinion polling for the 2022 Philippine presidential election ...
Scraping terminé pour Opinion polling for the 2022 Philippine presidential election en 0.4178 secondes.

Scraping : Opinion polling for the 2016 Philippine presidential election ...
Scraping terminé pour Opinion polling for the 2016 Philippine presidential election en 0.6986 secondes.

Scraping : Opinion polling for the 2010 Philippine presidential election ...
Scraping terminé pour Opinion polling for the 2010 Philippine presidential election en 0.4733 secondes.

Scraping : Opinion polling for the 2023 Polish parliamentary election ...
Scraping terminé pour Opinion polling for the 2023 Polish parliamentary election en 1.6063 secondes.

Scraping : Opinion polling for the 2019 Polish parliamentary election ...
Scraping terminé pour Opinion polling for the 2019 Polish parliamentary election en 1.7070 secondes.

Scraping : Opinion polling for the 2015 Polish parliamentary election ...
Scraping terminé pour Opinio

# **2. Nettoyage et traitement de données à l'aide de pandas**

# Manipulons nos tableaux par élection et par pays
## Pour chacune des élections, les deux tableaux de données sont fusionnés et les colonnnes d'intérêts retenues

#**The_2022_Philippine_presidential_election**

In [4]:
Opinion_polling_for_the_2022_Philippine_presidential_election_table_1 = pd.DataFrame(election_tables["Opinion_polling_for_the_2022_Philippine_presidential_election_table_1"])

Opinion_polling_for_the_2022_Philippine_presidential_election_table_2 = pd.DataFrame(election_tables["Opinion_polling_for_the_2022_Philippine_presidential_election_table_2"])

print(list(Opinion_polling_for_the_2022_Philippine_presidential_election_table_1.columns))

print(list(Opinion_polling_for_the_2022_Philippine_presidential_election_table_2.columns))

Philippine_2022_Presidential = pd.concat([Opinion_polling_for_the_2022_Philippine_presidential_election_table_1, Opinion_polling_for_the_2022_Philippine_presidential_election_table_2], join='inner', ignore_index=True)  # ignore_index réindexe proprement

Philippine_2022_Presidential = Philippine_2022_Presidential.drop(columns=['MoE', 'Ref.', 'Others', 'Und./None'], errors='ignore')

Philippine_2022_Presidential.columns = ['poll_date', 'polling_organisation', 'sample_size', 'Abella_Ind', 'De_Guzman_PLM', 'Gonzales_PDSP', 'Mangondato_Katipunan', 'Marcos_PFP', 'Montemayor_DPP', 'Moreno_Aksyon', 'Pacquiao_PROMDI', 'Robredo_Ind']

Philippine_2022_Presidential = Philippine_2022_Presidential.drop( 0 )

print(Philippine_2022_Presidential)


['Fieldworkdate(s)', 'Pollster', 'Samplesize', 'MoE', 'AbellaInd.', 'De GuzmanPLM', 'GonzalesPDSP', 'LacsonInd.[a]', 'MangondatoKatipunan', 'MarcosPFP', 'MontemayorDPP', 'MorenoAksyon', 'PacquiaoPROMDI', 'RobredoInd.', 'Others', 'Und./None', 'Ref.']
['Fieldworkdate(s)', 'Pollster', 'Samplesize', 'MoE', 'AbellaInd.', 'De GuzmanPLM', 'GonzalesPDSP', 'LacsonReporma', 'MangondatoKatipunan', 'MarcosPFP', 'MontemayorDPP', 'MorenoAksyon', 'PacquiaoPROMDI', 'RobredoInd.', 'Others', 'Und./None', 'Ref.']
        poll_date                               polling_organisation  \
1           May 9                                   Election results   
2       Exit poll                                   Publicus Asia[1]   
3         May 2–5                                   Publicus Asia[2]   
4       Apr 22–30                           Mobilis–TruthWatch[3][4]   
5       Apr 22–25                                         OCTA[5][6]   
6       Apr 19–21                                   Publicus Asia[7]

#**The_2016_Philippine_presidential_election**

In [5]:
Opinion_polling_for_the_2016_Philippine_presidential_election_table_1 = pd.DataFrame(election_tables["Opinion_polling_for_the_2016_Philippine_presidential_election_table_1"])

Opinion_polling_for_the_2016_Philippine_presidential_election_table_2 = pd.DataFrame(election_tables["Opinion_polling_for_the_2016_Philippine_presidential_election_table_2"])

print(list(Opinion_polling_for_the_2016_Philippine_presidential_election_table_1.columns))

print(list(Opinion_polling_for_the_2016_Philippine_presidential_election_table_2.columns))

Philippine_2016_Presidential = pd.concat([Opinion_polling_for_the_2016_Philippine_presidential_election_table_1, Opinion_polling_for_the_2016_Philippine_presidential_election_table_2], join='inner', ignore_index=True)  # ignore_index réindexe proprement

Philippine_2016_Presidential = Philippine_2016_Presidential.drop(columns=["MoE", "Refused", "Don't know"], errors='ignore')

Philippine_2016_Presidential.columns = ['polling_organisation','poll_date', 'sample_size', 'Binay_UNA', 'Poe_Ind', 'Duterte_PDP-Laban', 'Roxas_LP', 'Santiago_PRP']

Philippine_2016_Presidential = Philippine_2016_Presidential.drop( 0 )

print(Philippine_2016_Presidential)


['Polling firm', 'Fieldwork date', 'Sample size', 'MoE', 'BinayUNA', 'PoeInd', 'DutertePDP-Laban', 'RoxasLP', 'SantiagoPRP', 'Refused', "Don't know", 'Undecided / None']
['Polling firm', 'Fieldwork date', 'Sample size', 'MoE', 'BinayUNA', 'PoeInd', 'DutertePDP-Laban', 'RoxasLP', 'SantiagoPRP', 'DavidAK', 'SabioInd', 'SeñeresPMM', 'SyjucoInd', 'Others', 'Refused', "Don't know", 'None']
                                 polling_organisation           poll_date  \
1                                    Election results         May 9, 2016   
2                                              SWS[1]           Exit poll   
3                                     D' Strafford[2]       May 1–5, 2016   
4                                              SWS[3]       May 1–3, 2016   
5                                            Argus[4]     Apr 28–30, 2016   
6                                         Standard[5]  Apr 27-May 1, 2016   
7   Antonio Trillanes' exposé on Rodrigo Duterte's...                    

#**The_2010_Philippine_presidential_election**

In [6]:
Opinion_polling_for_the_2010_Philippine_presidential_election= pd.DataFrame(election_tables["Opinion_polling_for_the_2010_Philippine_presidential_election_table_1"])

print(list(Opinion_polling_for_the_2010_Philippine_presidential_election.columns))

Philippine_2010_Presidential = Opinion_polling_for_the_2010_Philippine_presidential_election.drop(columns=["MoE", "Others/Undecided"], errors='ignore')

Philippine_2010_Presidential.columns = ['polling_organisation','poll_date', 'sample_size', 'Acosta_KBL', 'Aquino_LP', 'De_los_Reyes_AKP', 'Estrada_PMP', 'GordonB_BAYAN', 'Madrigal_Ind.', 'Perlas_Ind.', 'Teodoro_LKS-KAM', 'Villanueva_BPP', 'Villar_NP',]

Philippine_2010_Presidential = Philippine_2010_Presidential.drop([0, 1])

print(Philippine_2010_Presidential)



['Polling firm', 'Fieldwork date', 'Sample size', 'MoE', 'AcostaKBL', 'AquinoLP', 'De los ReyesAKP', 'EstradaPMP', 'GordonB-BAYAN', 'MadrigalInd.', 'PerlasInd.', 'TeodoroLKS-KAM', 'VillanuevaBPP', 'VillarNP', 'Others/Undecided']
        polling_organisation       poll_date sample_size Acosta_KBL Aquino_LP  \
2           Election results           May 9  36,139,102          —     42.08   
3                     SWS[1]       Exit poll      52,573          —     43.34   
4                     SWS[2]         May 2–3       2,400          0        42   
5              StratPOLLS[3]   Apr. 27–May 2       1,500          —      45.2   
6              The Center[4]   Apr. 26–May 2       2,400          —        29   
7   Manila Standard Today[5]      Apr. 25–27       2,500          —        38   
8              Pulse Asia[6]      Apr. 23–25       1,800          —        39   
9   Manila Standard Today[7]      Apr. 18–20       2,500          —        38   
10                    SWS[8]      Apr. 16–

#**The_2023_Polish_parliamentary_election**

In [7]:
Opinion_polling_for_the_2023_Polish_parliamentary_election_table_1 = pd.DataFrame(election_tables["Opinion_polling_for_the_2023_Polish_parliamentary_election_table_1"])

Opinion_polling_for_the_2023_Polish_parliamentary_election_table_2 = pd.DataFrame(election_tables["Opinion_polling_for_the_2023_Polish_parliamentary_election_table_2"])

print(list(Opinion_polling_for_the_2023_Polish_parliamentary_election_table_1.columns))

print(list(Opinion_polling_for_the_2023_Polish_parliamentary_election_table_2.columns))

Polish_2023_parliamentary = pd.concat([Opinion_polling_for_the_2023_Polish_parliamentary_election_table_1, Opinion_polling_for_the_2023_Polish_parliamentary_election_table_2], join='inner', ignore_index=True)  # ignore_index réindexe proprement

Polish_2023_parliamentary  = Polish_2023_parliamentary .drop(columns=['Lead'], errors='ignore')

Polish_2023_parliamentary .columns = ['polling_organisation','poll_date', 'sample_size', 'United_Right', 'The_Left', 'Third_Way', 'Confederation']

Polish_2023_parliamentary = Polish_2023_parliamentary.drop([0, 1, 2,4] )

print(Polish_2023_parliamentary)


['Polling firm/Link', 'Fieldworkdate', 'Samplesize', 'UnitedRight', 'Civic Coalition', 'The Left', 'Third Way', 'Confederation', 'Nonpartisan Local Gov. Activists', 'There is One Poland', 'Others', "Don't know", 'Lead']
['Polling firm/Link', 'Fieldworkdate', 'Samplesize', 'UnitedRight', "Kukiz'15", 'CivicCoalition', 'AGROunia', 'Third Way', 'Confederation', 'The Left', 'Independents& Local Gov. Activists', "Others /Don't know", 'Lead']
              polling_organisation     poll_date sample_size United_Right  \
3           Parliamentary election        15 Oct  21,596,674        35.38   
5                              OGB        15 Oct                     33.5   
6                            IPSOS        15 Oct      90,000         36.8   
7                  Opinia24 / "GW"     12–13 Oct       1,500         32.0   
8             IBSP / Stan Polityki     11–13 Oct       1,100        37.36   
..                             ...           ...         ...          ...   
242  Social Changes /

#**The_2019_Polish_parliamentary_election**

In [8]:
Polish_2019_parliamentary = pd.DataFrame(election_tables["Opinion_polling_for_the_2019_Polish_parliamentary_election_table_1"])

print(list(Polish_2019_parliamentary.columns))

Polish_2019_parliamentary  = Polish_2019_parliamentary .drop(columns=["Others /Don't know", 'Lead'], errors='ignore')

Polish_2019_parliamentary .columns = ['polling_organisation','poll_date', 'sample_size', 'United_Right', 'Civic_Coalition', 'The_Left', 'Polish_Coalition', 'Confederation', 'Independents& Local Gov_Activists']

Polish_2019_parliamentary = Polish_2019_parliamentary.drop([0, 1] )

print(Polish_2019_parliamentary)


['Polling Firm/Link', 'FieldworkPeriod', 'Sample Size', 'United Right', 'Civic Coalition', 'The Left', 'Polish Coalition', 'Confederation', 'Independents& Local Gov. Activists', "Others /Don't know", 'Lead']
             polling_organisation       poll_date sample_size United_Right  \
2                         Results     13 Oct 2019                    43.59   
3                                                                            
4                           IPSOS     13 Oct 2019                     43.6   
5                     IBRiS / RMF     11 Oct 2019       1,100         42.0   
6                            IBSP  10-11 Oct 2019       1,002         42.1   
..                            ...             ...         ...          ...   
58  Social Changes / wpolityce.pl  26-31 Jul 2019       1,012         45.9   
59  Social Changes / wpolityce.pl  19-24 Jul 2019       1,064         44.8   
60                                                                           
61         P

#**The_2015_Polish_parliamentary_election**

In [9]:
Polish_2015_parliamentary = pd.DataFrame(election_tables["Opinion_polling_for_the_2015_Polish_parliamentary_election_table_1"])

print(list(Polish_2015_parliamentary.columns))

Polish_2015_parliamentary  = Polish_2015_parliamentary .drop(columns=['Others/Undecided', 'Lead'], errors='ignore')

Polish_2015_parliamentary .columns = ['poll_date','polling_organisation', 'PO', 'PiS', 'PSL', 'SLD', 'TR', "Kukiz'15", '.Nowoczesna', 'KORWiN', 'Razem']

Polish_2015_parliamentary = Polish_2015_parliamentary.drop([0] )

print(Polish_2015_parliamentary)


['Dates of Polling', 'Polling Firm/Link', 'PO', 'PiS', 'PSL', 'SLD', 'TR', "Kukiz'15", '.Nowoczesna', 'KORWiN', 'Razem', 'Others/Undecided', 'Lead']
           poll_date polling_organisation    PO   PiS  PSL   SLD   TR  \
1    25 October 2015     Election results  24.1  37.6  5.1   7.6  7.6   
2         25 October      Late pollsIpsos  23.6  37.7  5.2   7.5  7.5   
3         25 October      Exit pollsIpsos  23.4  39.1  5.2   6.6  6.6   
4      22–23 October                IBRiS  22.4  37.4  5.5   8.9  8.9   
5         22 October                IBRiS  24.4  35.3  4.7   8.1  8.1   
..               ...                  ...   ...   ...  ...   ...  ...   
135    16–17 January                IBRiS  35.6  34.7  7.8  10.2    1   
136     8–14 January                 CBOS    40    29    7     6    0   
137    12–13 January           TNS Poland    33    33    6     8    1   
138       12 January       Millward Brown    34    34    9     9    0   
139      2–3 January                IBRiS  33.9 

#**The_2011_Polish_parliamentary_election**

In [10]:
Polish_2011_parliamentary = pd.DataFrame(election_tables["Opinion_polling_for_the_2011_Polish_parliamentary_election_table_1"])

print(list(Polish_2011_parliamentary.columns))

Polish_2011_parliamentary  = Polish_2011_parliamentary .drop(columns=['Others/Undecided', 'Lead'], errors='ignore')

Polish_2011_parliamentary .columns = ['poll_date','polling_organisation', 'PO', 'PiS', 'SLD', 'PSL', 'RP', 'PJN']

Polish_2011_parliamentary = Polish_2011_parliamentary.drop(0 )

print(Polish_2011_parliamentary)

['Dates of Polling', 'Polling Firm/Link', 'PO', 'PiS', 'SLD', 'PSL', 'RP', 'PJN', 'Others/Undecided', 'Lead']
                poll_date polling_organisation    PO   PiS   SLD  PSL    RP  \
1         October 9, 2011     Election results  39.2  29.9   8.2  8.4  10.0   
2         October 7, 2011          Homo Homini  32.1  29.5  12.2  7.8   5.8   
3         October 4, 2011          GfK Polonia    46    31     9    5     4   
4       October 1–2, 2011              SMG/KRC    32    29    10    5     8   
5            October 2011                 CBOS    34    20     9    6     7   
6      September 29, 2011             TNS OBOP    31    22     6    6     7   
7      September 19, 2011              SMG/KRC    35    29    13    5     6   
8   September 16–19, 2011            Estymator  35.1  29.6  13.7  7.1   7.6   
9   September 14–15, 2011             TNS OBOP    40    33    11    7     5   
10  September 12–15, 2011            Estymator  36.3  28.6  14.9  7.1   6.3   
11      September 9, 

#**The_2007_Polish_parliamentary_election**

In [11]:
Polish_2007_parliamentary = pd.DataFrame(election_tables["Opinion_polling_for_the_2007_Polish_parliamentary_election_table_1"])

print(list(Polish_2007_parliamentary.columns))

Polish_2007_parliamentary  = Polish_2007_parliamentary .drop(columns=['Others', 'Undecided', 'Lead'], errors='ignore')

Polish_2007_parliamentary .columns = ['polling_organisation','poll_date', 'PiS', 'PO', 'SLD', 'UP', 'SDPL', 'PD', 'PSL', 'SRP', 'LPR']

Polish_2007_parliamentary = Polish_2007_parliamentary.drop(0)

print(Polish_2007_parliamentary)


['Polling Firm/Link', 'Last Dateof Polling', 'PiS', 'PO', 'SLD', 'UP', 'SDPL', 'PD', 'PSL', 'SRP', 'LPR', 'Others', 'Undecided', 'Lead']
   polling_organisation         poll_date   PiS    PO   SLD    UP  SDPL    PD  \
1      Election results  October 21, 2007  32.1  41.5  13.2  13.2  13.2  13.2   
2           GfK Polonia  October 19, 2007    35    42    12    12    12    12   
3                   PGB  October 19, 2007    31    35    17    17    17    17   
4              TNS OBOP  October 18, 2007    33    44    12    12    12    12   
5               PBS DGA  October 18, 2007    32    42    12    12    12    12   
..                  ...               ...   ...   ...   ...   ...   ...   ...   
63             TNS OBOP     March 5, 2007    22    34    11    11    11    11   
64                 CBOS  February 5, 2007    25    30    11    11    11    11   
65             TNS OBOP  February 5, 2007    28    31     6     -     3     1   
66                 CBOS  January 15, 2007    26    33

#**The_2005_Polish_parliamentary_election**

In [12]:
Polish_2005_parliamentary = pd.DataFrame(election_tables["Opinion_polling_for_the_2005_Polish_parliamentary_election_table_1"])

print(list(Polish_2005_parliamentary.columns))

Polish_2005_parliamentary  = Polish_2005_parliamentary .drop(columns=["Others /Don't know", 'Lead'], errors='ignore')

Polish_2005_parliamentary .columns = ['polling_organisation','poll_date', 'SLD', 'UP', 'SDPL', 'PO', 'PiS', 'PSL', 'SRP', 'LPR', 'PD']

Polish_2005_parliamentary = Polish_2005_parliamentary.drop(0 )

print(Polish_2005_parliamentary)

['Polling Firm/Link', 'Last Dateof Polling', 'SLD', 'UP', 'SDPL', 'PO', 'PiS', 'PSL', 'SRP', 'LPR', 'PD', "Others /Don't know", 'Lead']
   polling_organisation           poll_date         SLD          UP  \
1      Election results  September 25, 2005        11.3         3.9   
2          TNS OBOP/PBS  September 25, 2005        11.2         3.2   
3            Exit polls          Exit polls  Exit polls  Exit polls   
4              TNS OBOP  September 22, 2005         7.6         1.7   
5                   PBS  September 21, 2005           7           3   
..                  ...                 ...         ...         ...   
62                 CBOS    February 7, 2005           6           4   
63                  PGB    January 26, 2005           6           1   
64                 CBOS    January 10, 2005           6           3   
65             TNS OBOP    January 10, 2005           9           3   
66                  PBS     January 9, 2005          11           4   

          S

#**The_2001_Polish_parliamentary_election**

In [13]:
Polish_2001_parliamentary = pd.DataFrame(election_tables["Opinion_polling_for_the_2001_Polish_parliamentary_election_table_1"])

print(list(Polish_2001_parliamentary.columns))

Polish_2001_parliamentary  = Polish_2001_parliamentary .drop(columns=['Others / Undecided', 'Lead'], errors='ignore')

Polish_2001_parliamentary .columns = ['poll_date','polling_organisation', 'AWS', 'SLD', 'UP', 'UW', 'PSL', 'ROP', 'SRP', 'PO', 'PiS', 'LPR']

Polish_2001_parliamentary = Polish_2001_parliamentary.drop(0)

print(Polish_2001_parliamentary)

['Dates of Polling', 'Polling Firm/Link', 'AWS', 'SLD', 'UP', 'UW', 'PSL', 'ROP', 'SRP', 'PO', 'PiS', 'LPR', 'Others / Undecided', 'Lead']
            poll_date polling_organisation  AWS SLD  UP   UW PSL     ROP  \
1   23 September 2001     Election results  5.6  41  41  3.1   9  w. LPR   
2     19–20 September             TNS OBOP    5  46  46    3  11       -   
3        18 September             TNS OBOP    4  43  43    4  12       -   
4     17–18 September                  OBW    6  45  45    3  10       -   
5      September 2001                  PBS    8  47  47    5   9       -   
..                ...                  ...  ...  ..  ..  ...  ..     ...   
68      27–28 January                  PBS   14  46   5    5  10       -   
69            January             Demoskop    8  39   -    7  10       -   
70      13–15 January             TNS OBOP   12  45   3    7  14       5   
71      12–14 January               Pentor   13  50   7   10  12       -   
72        5–8 January    

#**Remodelage de toutes les bases de données**


## Le code ci-dessous transforme des bases de données électorales d'un format large (où chaque candidat a sa propre colonne) vers un format long (où chaque ligne représente une prédiction ou un résultat pour un candidat). Il identifie les variables d'identification (comme polling_organisation, poll_date, sample_size), sépare les résultats finaux des prédictions, puis les fusionne pour créer un nouveau dataframe standardisé pour chaque élection.


In [14]:
# Liste des noms de bases de données à traiter
database_names = [
    'Polish_2001_parliamentary', 'Polish_2005_parliamentary', 'Polish_2007_parliamentary',
    'Polish_2011_parliamentary', 'Polish_2015_parliamentary', 'Polish_2019_parliamentary',
    'Polish_2023_parliamentary', 'Philippine_2010_Presidential', 'Philippine_2016_Presidential',
    'Philippine_2022_Presidential'
]

# Fonction pour transformer une base de données
def transform_database(db_name, df):
    try:
        # Liste des variables d'identification possibles
        possible_id_vars = ['polling_organisation', 'poll_date', 'sample_size']

        # Identifier les variables d'identification présentes dans cette base
        id_vars = [col for col in possible_id_vars if col in df.columns]

        if not id_vars:
            print(f"Attention: Aucune variable d'identification reconnue dans {db_name}.")
            print(f"Colonnes disponibles: {df.columns.tolist()}")
            return None

        print(f"Variables d'identification utilisées pour {db_name}: {id_vars}")

        # Identifiez les colonnes qui sont des candidats
        candidate_cols = [col for col in df.columns if col not in id_vars]

        if not candidate_cols:
            print(f"Erreur: Aucune colonne de candidat trouvée dans {db_name}")
            return None

        # Utiliser l'index 0 pour les résultats
        result_idx = 0

        if len(df) == 0:
            print(f"Erreur: DataFrame vide pour {db_name}")
            return None

        print(f"Ligne de résultats trouvée à l'index {result_idx} pour {db_name}")

        # Créer un dataframe pour les résultats
        results_df = df.iloc[[result_idx]].copy()
        results_long = pd.melt(
            results_df,
            id_vars=id_vars,
            value_vars=candidate_cols,
            var_name='identity_candidate',
            value_name='result'
        )

        # Créer un dataframe pour les prévisions (toutes les lignes après la ligne de résultats)
        predictions_df = df.iloc[result_idx + 1:].copy()

        # Vérifier si nous avons des prédictions
        if predictions_df.empty:
            print(f"Attention: Pas de données de prédiction après la ligne de résultats dans {db_name}")
            return None

        predictions_long = pd.melt(
            predictions_df,
            id_vars=id_vars,
            value_vars=candidate_cols,
            var_name='identity_candidate',
            value_name='prediction'
        )

        # Fusionner les deux dataframes
        df_long = pd.merge(
            predictions_long,
            results_long[['identity_candidate', 'result']],
            on='identity_candidate',
            how='left'
        )

        print(f"Transformation réussie pour {db_name}. {len(df_long)} lignes générées.")
        return df_long

    except Exception as e:
        print(f"Erreur lors du traitement de {db_name}: {str(e)}")
        return None

# Traiter toutes les bases de données et les stocker sous les mêmes noms
successful_transformations = 0

for db_name in database_names:
    print(f"\nTraitement de {db_name}...")

    # Vérifier si la base de données existe dans l'espace global
    if db_name in globals():
        # Sauvegarder une copie de l'original au cas où
        original_df = globals()[db_name].copy()

        # Transformer la base de données
        transformed_df = transform_database(db_name, original_df)

        if transformed_df is not None:
            # Stocker la version transformée sous le même nom
            globals()[db_name] = transformed_df
            successful_transformations += 1

            print(f"Base de données {db_name} transformée et stockée avec succès.")
            print(f"Aperçu des données transformées:")
            print(globals()[db_name].head(3))
        else:
            print(f"Échec de la transformation pour {db_name}. La base originale est conservée.")
    else:
        print(f"Base de données {db_name} non trouvée dans l'espace de travail.")

print(f"\nTraitement terminé. {successful_transformations} bases de données transformées et stockées avec succès.")


Traitement de Polish_2001_parliamentary...
Variables d'identification utilisées pour Polish_2001_parliamentary: ['polling_organisation', 'poll_date']
Ligne de résultats trouvée à l'index 0 pour Polish_2001_parliamentary
Transformation réussie pour Polish_2001_parliamentary. 710 lignes générées.
Base de données Polish_2001_parliamentary transformée et stockée avec succès.
Aperçu des données transformées:
  polling_organisation        poll_date identity_candidate prediction result
0             TNS OBOP  19–20 September                AWS          5    5.6
1             TNS OBOP     18 September                AWS          4    5.6
2                  OBW  17–18 September                AWS          6    5.6

Traitement de Polish_2005_parliamentary...
Variables d'identification utilisées pour Polish_2005_parliamentary: ['polling_organisation', 'poll_date']
Ligne de résultats trouvée à l'index 0 pour Polish_2005_parliamentary
Transformation réussie pour Polish_2005_parliamentary. 585 lign

#**Affiliation politique de chaque candidat**


## Le code ci-dessous définit un dictionnaire nommé political_affiliations qui associe, pour chaque élection (Philippines et Pologne), les candidats ou partis politiques à leur orientation politique (gauche, centre, droite, extrême droite, etc.). Cette structure de données nous permettra d'attribuer automatiquement l'orientation politique à chaque candidat dans la suite du projet.

In [15]:
political_affiliations = {
    # Philippines 2022 Presidential
    "Philippine_2022_Presidential": {
        "Abella_Ind": "indépendant",
        "De_Guzman_PLM": "gauche",  # Parti des travailleurs, orientation socialiste
        "Gonzales_PDSP": "gauche",  # Parti démocrate-socialiste
        "Mangondato_Katipunan": "centre",
        "Marcos_PFP": "droite",  # Conservateur, nationaliste
        "Montemayor_DPP": "centre",
        "Moreno_Aksyon": "centre",
        "Pacquiao_PROMDI": "droite",  # Conservateur sur questions sociales
        "Robredo_Ind": "centre"  # Libérale, progressiste
    },

    # Philippines 2016 Presidential
    "Philippine_2016_Presidential": {
        "Binay_UNA": "droite",
        "Poe_Ind": "centre",
        "Duterte_PDP-Laban": "droite",  # Populiste, nationaliste
        "Roxas_LP": "centre",  # Libéral
        "Santiago_PRP": "centre"
    },

    # Philippines 2010 Presidential
    "Philippine_2010_Presidential": {
        "Acosta_KBL": "droite",  # Nationaliste
        "Aquino_LP": "centre",  # Libéral
        "De_los_Reyes_AKP": "gauche",
        "Estrada_PMP": "gauche",  # Populiste
        "GordonB_BAYAN": "gauche",
        "Madrigal_Ind.": "centre",
        "Perlas_Ind.": "vert",  # Écologiste
        "Teodoro_LKS-KAM": "droite",  # Conservateur
        "Villanueva_BPP": "droite",  # Chrétien-démocrate
        "Villar_NP": "droite"  # Nationaliste
    },

    # Pologne 2023 Parliamentary
    "Polish_2023_parliamentary": {
        "United_Right": "droite",  # Coalition conservatrice
        "The_Left": "gauche",  # Coalition de gauche
        "Third_Way": "centre",  # Coalition centriste
        "Confederation": "extrême droite"  # Nationaliste, libertarien
    },

    # Pologne 2019 Parliamentary
    "Polish_2019_parliamentary": {
        "United_Right": "droite",  # Coalition conservatrice
        "Civic_Coalition": "centre",  # Libéral
        "The_Left": "gauche",  # Coalition de gauche
        "Polish_Coalition": "droite",  # Agraire, chrétien-démocrate
        "Confederation": "extrême droite",  # Nationaliste, libertarien
        "Independents& Local Gov_Activists": "divers"
    },

    # Pologne 2015 Parliamentary
    "Polish_2015_parliamentary": {
        "PO": "droite",  # Plateforme civique, libéral-conservateur
        "PiS": "droite",  # Droit et Justice, conservateur
        "PSL": "droite",  # Parti paysan polonais, agraire
        "SLD": "gauche",  # Alliance de la gauche démocratique
        "TR": "gauche",  # Ton Mouvement, social-libéral
        "Kukiz'15": "droite",  # Populiste, anti-establishment
        ".Nowoczesna": "centre",  # Moderne, libéral
        "KORWiN": "extrême droite",  # Libertarien, conservateur
        "Razem": "gauche"  # Ensemble, socialiste
    },

    # Pologne 2011 Parliamentary
    "Polish_2011_parliamentary": {
        "PO": "droite",  # Plateforme civique
        "PiS": "droite",  # Droit et Justice
        "SLD": "gauche",  # Alliance de la gauche démocratique
        "PSL": "droite",  # Parti paysan polonais
        "RP": "centre",  # Mouvement Palikot, libéral
        "PJN": "droite"  # Pologne est la plus importante, conservateur
    },

    # Pologne 2007 Parliamentary
    "Polish_2007_parliamentary": {
        "PiS": "droite",  # Droit et Justice
        "PO": "droite",  # Plateforme civique
        "SLD": "gauche",  # Alliance de la gauche démocratique
        "UP": "gauche",  # Union du travail
        "SDPL": "gauche",  # Social-démocratie de Pologne
        "PD": "centre",  # Parti démocratique
        "PSL": "droite",  # Parti paysan polonais
        "SRP": "droite",  # Autodéfense de la République de Pologne
        "LPR": "extrême droite"  # Ligue des familles polonaises
    },

    # Pologne 2005 Parliamentary
    "Polish_2005_parliamentary": {
        "SLD": "gauche",  # Alliance de la gauche démocratique
        "UP": "gauche",  # Union du travail
        "SDPL": "gauche",  # Social-démocratie de Pologne
        "PO": "droite",  # Plateforme civique
        "PiS": "droite",  # Droit et Justice
        "PSL": "droite",  # Parti paysan polonais
        "SRP": "droite",  # Autodéfense de la République de Pologne
        "LPR": "extrême droite",  # Ligue des familles polonaises
        "PD": "centre"  # Parti démocratique
    },

    # Pologne 2001 Parliamentary
    "Polish_2001_parliamentary": {
        "AWS": "droite",  # Action électorale Solidarité
        "SLD": "gauche",  # Alliance de la gauche démocratique
        "UP": "gauche",  # Union du travail
        "UW": "centre",  # Union de la liberté
        "PSL": "droite",  # Parti paysan polonais
        "ROP": "droite",  # Mouvement pour la reconstruction de la Pologne
        "SRP": "droite",  # Autodéfense de la République de Pologne
        "PO": "droite",  # Plateforme civique
        "PiS": "droite",  # Droit et Justice
        "LPR": "extrême droite"  # Ligue des familles polonaises
    }
}

#**Indication de l'orientation politique de chaque candidat/parti**

## La fonction add_political_leaning_column ajoute une colonne d'orientation politique à chaque dataframe électoral en associant les candidats à leur affiliation politique définie dans le dictionnaire political_affiliations. Elle vérifie les correspondances manquantes, les remplace par "Non défini" si nécessaire, et retourne un dictionnaire contenant tous les dataframes mis à jour avec cette nouvelle information.

 #    Args:
   dataframes_dict (dict): Dictionnaire contenant les dataframes des élections (clés = noms des bases, valeurs = dataframes)
        political_affiliations (dict): Dictionnaire contenant les affiliations politiques par élection
        
#    Returns:
  dict: Dictionnaire mis à jour avec les dataframes contenant la nouvelle colonne
    

In [16]:
def add_political_leaning_column(dataframes_dict, political_affiliations):

    # Créer une copie du dictionnaire pour ne pas modifier l'original
    updated_dfs = {}

    # Parcourir chaque dataframe
    for db_name, df in dataframes_dict.items():
        # Vérifier si la base de données existe dans political_affiliations
        if db_name in political_affiliations:
            # Créer une copie du dataframe
            updated_df = df.copy()

            # Ajouter la colonne political_leaning_candidate
            updated_df['political_leaning_candidate'] = updated_df['identity_candidate'].map(
                political_affiliations[db_name]
            )

            # Si certains candidats n'ont pas de correspondance, afficher un avertissement
            if updated_df['political_leaning_candidate'].isna().any():
                missing_candidates = updated_df[updated_df['political_leaning_candidate'].isna()]['identity_candidate'].unique()
                print(f"Attention: Les candidats suivants de {db_name} n'ont pas d'affiliation politique définie: {', '.join(missing_candidates)}")
                # Remplacer les valeurs NaN par "Non défini"
                updated_df['political_leaning_candidate'] = updated_df['political_leaning_candidate'].fillna("Non défini")

            # Ajouter au dictionnaire de résultats
            updated_dfs[db_name] = updated_df
            print(f"Colonne 'political_leaning_candidate' ajoutée pour {db_name}")
        else:
            print(f"Attention: Aucune information politique trouvée pour {db_name}")
            updated_dfs[db_name] = df.copy()

    return updated_dfs

#**Application du code précédent sur nos données**

##Ce code crée un dictionnaire election_dataframes contenant tous les dataframes électoraux, applique la fonction add_political_leaning_column pour ajouter l'orientation politique à chaque candidat, puis remplace les dataframes originaux par leurs versions mises à jour avec cette nouvelle colonne d'affiliation politique.

In [17]:
# Créer un dictionnaire avec vos dataframes
election_dataframes = {
    "Polish_2001_parliamentary": Polish_2001_parliamentary,
    "Polish_2005_parliamentary": Polish_2005_parliamentary,
    "Polish_2007_parliamentary": Polish_2007_parliamentary,
    "Polish_2011_parliamentary": Polish_2011_parliamentary,
    "Polish_2015_parliamentary": Polish_2015_parliamentary,
    "Polish_2019_parliamentary": Polish_2019_parliamentary,
    "Polish_2023_parliamentary": Polish_2023_parliamentary,
    "Philippine_2010_Presidential": Philippine_2010_Presidential,
    "Philippine_2016_Presidential": Philippine_2016_Presidential,
    "Philippine_2022_Presidential": Philippine_2022_Presidential
}

# Appliquer la fonction pour ajouter la colonne political_leaning_candidate
updated_dataframes = add_political_leaning_column(election_dataframes, political_affiliations)

# Si vous souhaitez remplacer vos dataframes originaux par les versions mises à jour
Polish_2001_parliamentary = updated_dataframes["Polish_2001_parliamentary"]
Polish_2005_parliamentary = updated_dataframes["Polish_2005_parliamentary"]
Polish_2007_parliamentary = updated_dataframes["Polish_2007_parliamentary"]
Polish_2011_parliamentary = updated_dataframes["Polish_2011_parliamentary"]
Polish_2015_parliamentary = updated_dataframes["Polish_2015_parliamentary"]
Polish_2019_parliamentary = updated_dataframes["Polish_2019_parliamentary"]
Polish_2023_parliamentary = updated_dataframes["Polish_2023_parliamentary"]
Philippine_2010_Presidential = updated_dataframes["Philippine_2010_Presidential"]
Philippine_2016_Presidential = updated_dataframes["Philippine_2016_Presidential"]
Philippine_2022_Presidential = updated_dataframes["Philippine_2022_Presidential"]

#Affichage des données
Polish_2001_parliamentary.head(20)
Polish_2001_parliamentary.head(20)
Polish_2005_parliamentary.head(20)
Polish_2007_parliamentary.head(20)
Polish_2011_parliamentary.head(20)
Polish_2015_parliamentary.head(20)
Polish_2019_parliamentary.head(20)
Polish_2023_parliamentary.head(20)
Philippine_2010_Presidential.head(20)
Philippine_2016_Presidential.head(20)
Philippine_2022_Presidential.head(20)

Colonne 'political_leaning_candidate' ajoutée pour Polish_2001_parliamentary
Colonne 'political_leaning_candidate' ajoutée pour Polish_2005_parliamentary
Colonne 'political_leaning_candidate' ajoutée pour Polish_2007_parliamentary
Colonne 'political_leaning_candidate' ajoutée pour Polish_2011_parliamentary
Colonne 'political_leaning_candidate' ajoutée pour Polish_2015_parliamentary
Colonne 'political_leaning_candidate' ajoutée pour Polish_2019_parliamentary
Colonne 'political_leaning_candidate' ajoutée pour Polish_2023_parliamentary
Colonne 'political_leaning_candidate' ajoutée pour Philippine_2010_Presidential
Colonne 'political_leaning_candidate' ajoutée pour Philippine_2016_Presidential
Colonne 'political_leaning_candidate' ajoutée pour Philippine_2022_Presidential


,polling_organisation,poll_date,sample_size,identity_candidate,prediction,result,political_leaning_candidate
0,Publicus Asia[1],Exit poll,"29,024",Abella_Ind,—,0.21,indépendant
1,Publicus Asia[2],May 2–5,"1,500",Abella_Ind,1,0.21,indépendant
2,Mobilis–TruthWatch[3][4],Apr 22–30,"2,400",Abella_Ind,—,0.21,indépendant
3,OCTA[5][6],Apr 22–25,"2,400",Abella_Ind,—,0.21,indépendant
4,Publicus Asia[7],Apr 19–21,"1,500",Abella_Ind,1,0.21,indépendant
5,Pulse Asia[8],Apr 16–21,"2,400",Abella_Ind,1,0.21,indépendant
6,Laylo[9][10],Apr 14–20,"3,000",Abella_Ind,—,0.21,indépendant
7,MBC–DZRH[11],Apr 18–19,"7,560",Abella_Ind,0.1,0.21,indépendant
8,I&AC[12],Apr 4–15,"2,440",Abella_Ind,0.375,0.21,indépendant
9,OCTA[13],Apr 2–6,"1,200",Abella_Ind,1,0.21,indépendant


#**Détection des outliers**

Cette fonction identifie les valeurs statistiquement aberrantes dans les données électorales en utilisant principalement la méthode du score Z. Elle parcourt chaque dataframe et chaque colonne numérique pour détecter les valeurs qui s'écartent significativement de la moyenne (au-delà d'un seuil défini, ici de 3 écarts-types), marque les lignes contenant des valeurs aberrantes avec un indicateur has_outlier. Les résultats sont conservés dans les dataframes originaux tout en marquant les outliers

   Paramètres:
    - final_cleaned_dataframes: Dictionnaire de DataFrames déjà nettoyés
    - method: Méthode de détection des outliers ('zscore')
    - threshold: Seuil pour la détection (3)

  Retourne:
    - Dictionnaire de DataFrames avec outliers identifiés
    - Résumé des outliers détectés
    

In [18]:
def detect_and_handle_outliers(updated_dataframes, method='zscore', threshold=3):

    final_dataframes = {}
    outlier_summary = {}

    for name, df in updated_dataframes.items():
        print(f"\nAnalyse des outliers pour {name}:")

        # Copie du DataFrame pour éviter de modifier l'original
        df_clean = df.copy()

        # 1. Identifier les colonnes numériques (celles qui contiennent des pourcentages électoraux)
        non_numeric_cols = ['poll_date', 'polling_organisation', 'sample_size', 'identity_candidate', 'political_leaning_candidate']
        # Filtrer pour ne garder que les colonnes qui existent réellement dans le dataframe
        existing_non_numeric = [col for col in non_numeric_cols if col in df_clean.columns]
        numeric_cols = [col for col in df_clean.columns if col not in existing_non_numeric]

        # 2. Convertir les colonnes numériques en type numérique si ce n'est pas déjà le cas
        for col in numeric_cols:
            df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

        # 3. Créer un DataFrame pour stocker les informations sur les outliers
        outlier_info = pd.DataFrame(index=df_clean.index)

        # 4. Détecter les outliers pour chaque colonne numérique
        for col in numeric_cols:
            # Ignorer les colonnes avec trop de valeurs manquantes
            if df_clean[col].isna().sum() > len(df_clean) * 0.5:
                continue

            if method == 'zscore':
                # Méthode Z-score
                mean = df_clean[col].mean()
                std = df_clean[col].std()

                if std == 0:  # Éviter division par zéro
                    continue

                z_scores = (df_clean[col] - mean) / std
                outliers = abs(z_scores) > threshold #prend la valeur absolue du score Z
                outlier_info[f'{col}_outlier'] = outliers

                print(f"  Colonne {col}: {outliers.sum()} outliers détectés")
                print(f"    Limites Z-score: [-{threshold}, {threshold}]")


            else:
                raise ValueError(f"Méthode de détection '{method}' non reconnue. Utilisez 'zscore'.")

        # 5. Ajouter une colonne indiquant si une ligne contient au moins un outlier
        if not outlier_info.empty:
            outlier_info['has_outlier'] = outlier_info.any(axis=1)

            # 6. Fusionner l'information sur les outliers avec le DataFrame original
            df_clean['has_outlier'] = outlier_info['has_outlier']

            # 7. Afficher des statistiques sur les outliers
            total_outliers = outlier_info['has_outlier'].sum()
            print(f"Total des lignes avec au moins un outlier: {total_outliers} ({total_outliers/len(df_clean)*100:.2f}%)")

            # 8. Créer un résumé des outliers pour ce DataFrame
            outlier_summary[name] = {
                'total_rows': len(df_clean),
                'rows_with_outliers': total_outliers,
                'percentage': total_outliers/len(df_clean)*100 if len(df_clean) > 0 else 0,
                'outliers_by_column': {col: outlier_info[f'{col}_outlier'].sum() for col in numeric_cols
                                      if f'{col}_outlier' in outlier_info.columns}
            }
        else:
            # Cas où aucune colonne numérique n'a été analysée
            df_clean['has_outlier'] = False
            outlier_summary[name] = {
                'total_rows': len(df_clean),
                'rows_with_outliers': 0,
                'percentage': 0,
                'outliers_by_column': {}
            }
            print("Aucun outlier détecté ou aucune colonne numérique valide pour l'analyse.")

        # 9. Stocker les différentes versions du DataFrame
        final_dataframes[name] = {
            'original': df_clean,  # Avec marquage des outliers
            # Vous pourriez ajouter d'autres versions ici, comme:
            # 'no_outliers': df_clean[~df_clean['has_outlier']],  # Sans les outliers
            # 'capped': df_capped  # Avec les outliers plafonnés
        }

    # 10. Créer un rapport de synthèse sur les outliers
    print("\nRapport de synthèse sur les outliers:")
    for name, summary in outlier_summary.items():
        print(f"{name}: {summary['rows_with_outliers']} lignes avec outliers ({summary['percentage']:.2f}%)")
        if summary['outliers_by_column']:
            print("  Détail par colonne:")
            for col, count in summary['outliers_by_column'].items():
                print(f"    {col}: {count} outliers")

    return final_dataframes, outlier_summary


outlier_dataframes, outlier_summary = detect_and_handle_outliers(updated_dataframes, method='zscore', threshold=3)


# Afficher les premières lignes de chaque DataFrame original (avec outliers identifiés)
print ( "\nAperçu des DataFrames avec outliers identifiés pour toutes les élections:" )
for Election_name, df_dict in outlier_dataframes.items():
     print ( f"\n { '-' * 80 } \n {Election_name} :\n { '-' * 80 } " )
     print (df_dict[ 'original' ].head())


Analyse des outliers pour Polish_2001_parliamentary:
  Colonne prediction: 0 outliers détectés
    Limites Z-score: [-3, 3]
  Colonne result: 0 outliers détectés
    Limites Z-score: [-3, 3]
Total des lignes avec au moins un outlier: 0 (0.00%)

Analyse des outliers pour Polish_2005_parliamentary:
  Colonne prediction: 3 outliers détectés
    Limites Z-score: [-3, 3]
  Colonne result: 0 outliers détectés
    Limites Z-score: [-3, 3]
Total des lignes avec au moins un outlier: 3 (0.51%)

Analyse des outliers pour Polish_2007_parliamentary:
  Colonne prediction: 0 outliers détectés
    Limites Z-score: [-3, 3]
  Colonne result: 0 outliers détectés
    Limites Z-score: [-3, 3]
Total des lignes avec au moins un outlier: 0 (0.00%)

Analyse des outliers pour Polish_2011_parliamentary:
  Colonne prediction: 0 outliers détectés
    Limites Z-score: [-3, 3]
  Colonne result: 0 outliers détectés
    Limites Z-score: [-3, 3]
Total des lignes avec au moins un outlier: 0 (0.00%)

Analyse des outlier

#**Création d'un fichier Excel pour chaque élection**

##Le code suivant extrait les informations de chaque nom d'élection (pays, année, type). Il crée ensuite des fichiers Excel standardisés pour chaque élection dans un format uniforme avec des colonnes pour la date du sondage, l'organisation de sondage, et pour chaque candidat: prédiction, résultat final, identité et orientation politique. Enfin, il sauvegarde ces données standardisées dans des fichiers Excel organisés dans un répertoire "election_files".

In [19]:
# Liste des élections
elections = [
    'Polish_2001_parliamentary', 'Polish_2005_parliamentary', 'Polish_2007_parliamentary',
    'Polish_2011_parliamentary', 'Polish_2015_parliamentary', 'Polish_2019_parliamentary',
    'Polish_2023_parliamentary', 'Philippine_2010_Presidential', 'Philippine_2016_Presidential',
    'Philippine_2022_Presidential'
]

# Fonction pour extraire les informations du nom de l'élection
def extract_info(election_name):
    parts = election_name.split('_')
    country = parts[0]
    year = parts[1]
    election_type = parts[2]
    return country, year, election_type

# Fonction pour créer un fichier Excel standardisé pour chaque élection
def create_standardized_excel(election_name, election_data, output_dir='election_files'):
    # Extraire les informations du nom de l'élection
    country, year, election_type = extract_info(election_name)

    # Créer le répertoire de sortie s'il n'existe pas
    os.makedirs(output_dir, exist_ok=True)

    # Nom du fichier standardisé
    filename = f"{country}_{year}_{election_type}.xlsx"
    filepath = os.path.join(output_dir, filename)

    # Vérifier la structure de election_data
    print(f"Type de données pour {election_name}: {type(election_data)}")

    # Si election_data est un dictionnaire, essayons de trouver le DataFrame
    if isinstance(election_data, dict):
        # Chercher un DataFrame dans les valeurs du dictionnaire
        df_found = False
        for key, value in election_data.items():
            if isinstance(value, pd.DataFrame):
                election_df = value
                df_found = True
                print(f"DataFrame trouvé sous la clé '{key}' pour {election_name}")
                break

        if not df_found:
            # Si aucun DataFrame n'est trouvé directement, cherchons plus profondément
            for key, value in election_data.items():
                if isinstance(value, dict):
                    for subkey, subvalue in value.items():
                        if isinstance(subvalue, pd.DataFrame):
                            election_df = subvalue
                            df_found = True
                            print(f"DataFrame trouvé sous les clés '{key}.{subkey}' pour {election_name}")
                            break
                if df_found:
                    break

        if not df_found:
            print(f"Aucun DataFrame trouvé pour {election_name}")
            # Créer un DataFrame vide avec les colonnes attendues
            election_df = pd.DataFrame(columns=[
                'poll_date', 'polling_organisation', 'identity_candidate',
                'prediction', 'result', 'political_leaning_candidate'
            ])
    elif isinstance(election_data, pd.DataFrame):
        election_df = election_data
    else:
        print(f"Type de données non pris en charge pour {election_name}: {type(election_data)}")
        return

    # Afficher les colonnes disponibles pour le débogage
    print(f"Colonnes disponibles pour {election_name}: {election_df.columns.tolist()}")

    # Vérifier si les colonnes nécessaires existent
    required_columns = ['poll_date', 'polling_organisation', 'identity_candidate', 'prediction', 'result', 'political_leaning_candidate']
    missing_columns = [col for col in required_columns if col not in election_df.columns]

    if missing_columns:
        print(f"Colonnes manquantes pour {election_name}: {missing_columns}")
        # Ajouter les colonnes manquantes avec des valeurs NaN
        for col in missing_columns:
            election_df[col] = pd.NA

    # Identifier les candidats uniques
    candidates = election_df['identity_candidate'].unique()
    print(f"Candidats identifiés pour {election_name}: {candidates}")

    # Créer un nouveau DataFrame pour stocker les données restructurées
    new_df = pd.DataFrame()

    # Ajouter les colonnes communes
    new_df['poll_date'] = election_df['poll_date'].unique()

    # Vérifier si sample_size existe et l'ajouter si disponible
    if 'sample_size' in election_df.columns:
        # Supposons que sample_size est le même pour tous les candidats d'un même sondage
        sample_sizes = {}
        for date in new_df['poll_date']:
            date_df = election_df[election_df['poll_date'] == date]
            if not date_df.empty:
                sample_sizes[date] = date_df['sample_size'].iloc[0]
            else:
                sample_sizes[date] = None
        new_df['sample_size'] = [sample_sizes.get(date) for date in new_df['poll_date']]

    # Ajouter polling_organisation si disponible
    orgs = {}
    for date in new_df['poll_date']:
        date_df = election_df[election_df['poll_date'] == date]
        if not date_df.empty:
            orgs[date] = date_df['polling_organisation'].iloc[0]
        else:
            orgs[date] = None
    new_df['polling_organization'] = [orgs.get(date) for date in new_df['poll_date']]

    # Préparer les colonnes pour tous les candidats d'abord
    for i, candidate in enumerate(candidates, 1):
        candidate_df = election_df[election_df['identity_candidate'] == candidate]

        # Initialiser les colonnes pour ce candidat
        new_df[f'prediction_result_candidate{i}'] = None
        new_df[f'final_result_candidate{i}'] = None
        new_df[f'identity_candidate{i}'] = candidate
        new_df[f'political_leaning_candidate{i}'] = None

        if not candidate_df.empty:
            # Résultat final (le même pour toutes les lignes)
            new_df[f'final_result_candidate{i}'] = candidate_df['result'].iloc[0]
            # Orientation politique (la même pour toutes les lignes)
            new_df[f'political_leaning_candidate{i}'] = candidate_df['political_leaning_candidate'].iloc[0]

            # Pour chaque date de sondage, trouver la prédiction pour ce candidat
            for idx, date in enumerate(new_df['poll_date']):
                candidate_date_df = candidate_df[candidate_df['poll_date'] == date]
                if not candidate_date_df.empty:
                    new_df.at[idx, f'prediction_result_candidate{i}'] = candidate_date_df['prediction'].iloc[0]

    # Réorganiser les colonnes dans l'ordre spécifié
    columns = ['poll_date']

    # Ajouter sample_size si disponible
    if 'sample_size' in new_df.columns:
        columns.append('sample_size')

    # Ajouter polling_organization si disponible
    if 'polling_organization' in new_df.columns:
        columns.append('polling_organization')

    # Ajouter les colonnes pour chaque candidat dans l'ordre spécifié
    for i in range(1, len(candidates) + 1):
        columns.extend([
            f'prediction_result_candidate{i}',
            f'final_result_candidate{i}',
            f'identity_candidate{i}',
            f'political_leaning_candidate{i}'
        ])

    # Réorganiser le DataFrame
    new_df = new_df[columns]

    # Sauvegarder le DataFrame restructuré dans un fichier Excel
    new_df.to_excel(filepath, index=False)
    print(f"Fichier créé: {filepath}")

# Vérifier si outlier_dataframes existe, sinon le créer comme un dictionnaire vide
try:
    outlier_dataframes
except NameError:
    print("Variable 'outlier_dataframes' non définie. Création d'un dictionnaire vide.")
    outlier_dataframes = {}

# Créer un fichier Excel standardisé pour chaque élection
for election_name in elections:
    if election_name in outlier_dataframes:
        try:
            create_standardized_excel(election_name, outlier_dataframes[election_name])
        except Exception as e:
            print(f"Erreur lors du traitement de {election_name}: {str(e)}")
    else:
        print(f"Attention: {election_name} n'est pas présent dans outlier_dataframes")

# Création et modification des fichiers durant l'execution:
print("Fichiers créés:")
for election_name in elections:
    country, year, election_type = extract_info(election_name)
    print(f"election_files/{country}_{year}_{election_type}.xlsx")

Type de données pour Polish_2001_parliamentary: <class 'dict'>
DataFrame trouvé sous la clé 'original' pour Polish_2001_parliamentary
Colonnes disponibles pour Polish_2001_parliamentary: ['polling_organisation', 'poll_date', 'identity_candidate', 'prediction', 'result', 'political_leaning_candidate', 'has_outlier']
Candidats identifiés pour Polish_2001_parliamentary: ['AWS' 'SLD' 'UP' 'UW' 'PSL' 'ROP' 'SRP' 'PO' 'PiS' 'LPR']
Fichier créé: election_files/Polish_2001_parliamentary.xlsx
Type de données pour Polish_2005_parliamentary: <class 'dict'>
DataFrame trouvé sous la clé 'original' pour Polish_2005_parliamentary
Colonnes disponibles pour Polish_2005_parliamentary: ['polling_organisation', 'poll_date', 'identity_candidate', 'prediction', 'result', 'political_leaning_candidate', 'has_outlier']
Candidats identifiés pour Polish_2005_parliamentary: ['SLD' 'UP' 'SDPL' 'PO' 'PiS' 'PSL' 'SRP' 'LPR' 'PD']
Fichier créé: election_files/Polish_2005_parliamentary.xlsx
Type de données pour Polish

#**Standardisation des dates et gestion des valeurs manquantes**

In [20]:
def standardize_poll_date(df):
       def clean_date1(date_str):
        # Remplacer les underscores par des espaces
        date_str = date_str.replace("_", " ")

        # Extraire les nombres présents dans la chaîne
        numbers = re.findall(r'\d+', date_str)

        # Garder les deux derniers nombres si plus d'un est trouvé, sinon garder le seul nombre trouvé
        if len(numbers) >= 2:
            numbers = numbers[-2:]
        elif len(numbers) == 1:
            numbers = numbers

        # Extraire les trois premières lettres du mois si présentes
        text_match = re.search(r'[a-zA-Z]{3}', date_str)
        month_abbr = text_match.group(0) if text_match else ""

        # Construire la nouvelle date formatée
        standardized_date = " ".join([month_abbr] + numbers)

        return standardized_date.strip()

    # Appliquer la transformation sur la colonne "poll_date"
       if "poll_date" in df.columns:
        df["poll_date"] = df["poll_date"].astype(str).apply(clean_date1)

       return df

In [21]:
def clean_poll_date(df, df_name):
    # Extraire l'année (4 chiffres) du nom du dataframe
    year_match = re.search(r'\d{4}', df_name)
    year = year_match.group(0) if year_match else "Unknown"

    def clean_date2(date_str):
        # Vérifier si la valeur est valide
        if not isinstance(date_str, str) or date_str.strip() == "":
            return date_str

        # Compter les espaces
        space_count = date_str.count(" ")

        # Si la ligne contient deux espaces, supprimer les nombres après le deuxième espace
        if space_count == 2:
            parts = date_str.split(" ")
            date_str = " ".join(parts[:2])  # Garder uniquement les deux premiers éléments

        # Ajouter l'année extraite du nom du dataframe
        date_str = f"{date_str} {year}"
        return date_str.strip()

    # Appliquer la transformation sur la colonne "poll_date"
    if "poll_date" in df.columns:
        df["poll_date"] = df["poll_date"].astype(str).apply(clean_date2)
    return df

In [22]:
# Fonction pour nettoyer la colonne "polling_organization" et remplacer les valeurs manquantes
def clean_poll_org(df):
    """
    Fonction pour nettoyer la colonne "polling_organization" et remplacer les valeurs manquantes dans tout le dataframe.

    Args:
        df (pd.DataFrame): Le dataframe contenant la colonne "polling_organization".

    Returns:
        pd.DataFrame: Le dataframe avec la colonne nettoyée et les valeurs manquantes remplacées par 0.
    """
    def extract_org_name(org_str):
        """Extrait le nom de l'organisation avant '/' ou '[' """
        if not isinstance(org_str, str):
            return org_str  # Retourner tel quel si ce n'est pas une chaîne
        return re.split(r'\/|\[', org_str)[0].strip()  # Prendre avant '/' ou '['

    # Appliquer la transformation sur "polling_organization"
    if "polling_organization" in df.columns:
        df["polling_organization"] = df["polling_organization"].astype(str).apply(extract_org_name)

    # Remplacer toutes les valeurs manquantes par 0
    df.fillna(0, inplace=True)

    return df

In [23]:
def organize_data(df):
    """
    - Vérifie et nettoie la colonne 'poll_date'.
    - Convertit 'poll_date' en format datetime.
    - Trie les données par date.
    - Retourne le DataFrame modifié.
    """
    # Supprimer les espaces en trop et uniformiser les dates
    df['poll_date'] = df['poll_date'].astype(str).str.strip()

    # Vérifier si la colonne contient déjà des dates au format différent
    df['poll_date'] = pd.to_datetime(df['poll_date'], errors='coerce')
    # Trier par date après conversion
    df = df.sort_values(by='poll_date')
    df = df.dropna()
    return df

In [24]:
def date_manipulate(df, file_name):
    df = standardize_poll_date(df)
    df = clean_poll_date(df, file_name)
    df = clean_poll_org(df)
    df = organize_data(df)
    return df


In [25]:
data = {
        "2022": "election_files/Philippine_2022_Presidential.xlsx",
        "2016": "election_files/Philippine_2016_Presidential.xlsx",
        "2010": "election_files/Philippine_2010_Presidential.xlsx",
        "2023": "election_files/Polish_2023_parliamentary.xlsx",
        "2019": "election_files/Polish_2019_parliamentary.xlsx",
        "2015": "election_files/Polish_2015_parliamentary.xlsx",
        "2011": "election_files/Polish_2011_parliamentary.xlsx",
        "2007": "election_files/Polish_2007_parliamentary.xlsx",
        "2005": "election_files/Polish_2005_parliamentary.xlsx",
        "2001": "election_files/Polish_2001_parliamentary.xlsx"
}


# Boucle pour lire chaque fichier Excel
for year, file_name in data.items():
    try:
        # Lire le fichier Excel dans un DataFrame
        df = pd.read_excel(file_name)
        df = date_manipulate(df, file_name)
        df.to_excel(file_name, index=False)
        # Afficher les premières lignes du DataFrame
        print(f"{year}: Lecture terminée pour {file_name}")
        print(df.head())
    except Exception as e:
        print(f"Erreur lors de la lecture du fichier {file_name}: {e}")


<ipython-input-23-f0a083d8b639>:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['poll_date'] = pd.to_datetime(df['poll_date'], errors='coerce')
<ipython-input-23-f0a083d8b639>:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['poll_date'] = pd.to_datetime(df['poll_date'], errors='coerce')
<ipython-input-23-f0a083d8b639>:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['poll_date'] = pd.to_datetime(df['poll_date'], errors='coerce')


2022: Lecture terminée pour election_files/Philippine_2022_Presidential.xlsx
    poll_date sample_size polling_organization  prediction_result_candidate1  \
36 2022-01-10      15,450                Laylo                          0.00   
39 2022-01-10       2,400                 I&AC                          0.25   
38 2022-01-17       3,000                Laylo                          0.00   
37 2022-01-19       2,400           Pulse Asia                          0.05   
35 2022-01-22      10,000               RP-MDF                          0.34   

    final_result_candidate1 identity_candidate1 political_leaning_candidate1  \
36                     0.21          Abella_Ind                  indépendant   
39                     0.21          Abella_Ind                  indépendant   
38                     0.21          Abella_Ind                  indépendant   
37                     0.21          Abella_Ind                  indépendant   
35                     0.21          Abell

<ipython-input-23-f0a083d8b639>:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['poll_date'] = pd.to_datetime(df['poll_date'], errors='coerce')


2015: Lecture terminée pour election_files/Polish_2015_parliamentary.xlsx
     poll_date polling_organization  prediction_result_candidate1  \
127 2015-01-02                IBRiS                          33.9   
124 2015-01-08                 CBOS                          40.0   
126 2015-01-12       Millward Brown                          34.0   
125 2015-01-12           TNS Poland                          33.0   
122 2015-01-15          GfK Polonia                          38.5   

     final_result_candidate1 identity_candidate1 political_leaning_candidate1  \
127                     24.1                  PO                       droite   
124                     24.1                  PO                       droite   
126                     24.1                  PO                       droite   
125                     24.1                  PO                       droite   
122                     24.1                  PO                       droite   

     prediction_result_c

Ce code prépare le dataframe pour l'analyse et la visualisation en appliquant plusieurs transformations :
    - Supprime les colonnes inutiles.
    - Stocke les valeurs uniques des colonnes d'identité et d'affiliation politique.
    - Remplace temporairement ces valeurs par des identifiants numériques.
    - Agrège les données par "poll_date".
    - Restaure les valeurs textuelles d'origine.
 Il prend en argument Le dataframe d'entrée (pd.DataFrame) et retourne le dataframe transformé.
    

In [26]:
# Fonction pour calculer la moyenne sur les dates

def prepare_data_for_graph(df):

    # Supprimer les valeurs manquantes
    df = df.dropna()

    # Supprimer les colonnes inutiles si elles existent
    columns_to_remove = ["polling_organization", "sample_size"]
    df = df.drop(columns=[col for col in columns_to_remove if col in df.columns], errors="ignore")

    # Trouver les colonnes correspondant aux identités des candidats et affiliations politiques
    identity_cols = [col for col in df.columns if col.lower().startswith("identity")]
    political_cols = [col for col in df.columns if col.lower().startswith("political_leaning")]

    # Vérifier si ces colonnes existent
    if not identity_cols or not political_cols:
        raise ValueError("Colonnes d'identité ou d'affiliation politique non trouvées dans le DataFrame.")

    # Stocker les valeurs uniques des identités et affiliations politiques
    identity_dict = {col: df[col].iloc[0] for col in identity_cols}
    political_leaning_dict = {col: df[col].iloc[0] for col in political_cols}

    # Remplacement temporaire des valeurs des colonnes identitaires et politiques par des identifiants numériques
    for col in identity_cols + political_cols:
        df[col] = 0  # Remplacement par une valeur neutre pour permettre l'agrégation

    # Agrégation des données par "poll_date"
    df = df.groupby("poll_date", as_index=False).mean(numeric_only=True)

    # Restauration des valeurs d'identité et d'affiliation politique
    for col in identity_cols:
        df[col] = identity_dict[col]

    for col in political_cols:
        df[col] = political_leaning_dict[col]

    return df

########
def clean_prediction(df):
    """
    Supprime les lignes contenant des valeurs 0 dans les colonnes commençant par 'prediction'.

    Arguments:
    df -- DataFrame pandas

    Retourne:
    df_cleaned -- DataFrame nettoyé
    """
    # Sélectionner les colonnes qui commencent par 'prediction'
    prediction_cols = [col for col in df.columns if col.startswith("prediction")]

    # Supprimer les lignes où au moins une colonne 'prediction' contient 0
    df_cleaned = df[~(df[prediction_cols] == 0).any(axis=1)].reset_index()

    return df_cleaned

########

def remove_columns_before_zero_final(df):
    """
    Supprime la colonne précédente à une colonne 'final' si la moyenne de cette colonne 'final' est 0.
    Cette fonction est générique et fonctionne avec tout DataFrame ayant des colonnes "final".

    Arguments:
    df -- DataFrame pandas

    Retourne:
    df_cleaned -- DataFrame nettoyé
    """
    # Identifier les colonnes qui commencent par "final"
    final_cols = [col for col in df.columns if col.startswith("final")]

    # Identifier les colonnes à supprimer
    cols_to_drop = []

    for final_col in final_cols:
        if df[final_col].mean() == 0:  # Vérifie si la moyenne est nulle
            final_col_index = df.columns.get_loc(final_col)
            if final_col_index > 0:  # Vérifie qu'il y a une colonne avant
                prev_col = df.columns[final_col_index - 1]
                cols_to_drop.append(prev_col)

    # Supprimer les colonnes identifiées
    df_cleaned = df.drop(columns=cols_to_drop, errors='ignore')

    return df_cleaned



# Charger les fichiers disponibles
data_files = {
    "Philippine": {
        "2022": "election_files/Philippine_2022_Presidential.xlsx",
        "2016": "election_files/Philippine_2016_Presidential.xlsx",
        "2010": "election_files/Philippine_2010_Presidential.xlsx"
    },
    "Poland" : {
        "2023": "election_files/Polish_2023_parliamentary.xlsx",
        "2019": "election_files/Polish_2019_parliamentary.xlsx",
        "2015": "election_files/Polish_2015_parliamentary.xlsx",
        "2011": "election_files/Polish_2011_parliamentary.xlsx",
        "2007": "election_files/Polish_2007_parliamentary.xlsx",
        "2005": "election_files/Polish_2005_parliamentary.xlsx",
        "2001": "election_files/Polish_2001_parliamentary.xlsx",
    }
}

#**Création de l'interface utilisateur**

In [ ]:
import streamlit as st  # interface utilisateur interactive
import matplotlib.pyplot as plt  # visualisation des données

# Interface utilisateur
st.title("📊Data Scrapers: Évolution des prédictions avant les élections: Philippine et Poland")

# Disposition en colonnes : 2 colonnes (Menu à gauche, Graphique à droite)
col1, col2 = st.columns([1, 3])

with col1:
    st.subheader("🔧 Paramètres")

    # Sélection du pays
    selected_country = st.selectbox("🌍 Sélectionnez un pays :", list(data_files.keys()))

    # Sélection de l'année des élections
    selected_year = st.selectbox("📅 Sélectionnez une année d'élection :", list(data_files[selected_country].keys()))

    # Charger les données
    file_path = f"{data_files[selected_country][selected_year]}"

    if os.path.exists(file_path):
        df = pd.read_excel(file_path)
        data = df
        # Préparation et transformation des données pour le graphique
        df = prepare_data_for_graph(df)
        df = remove_columns_before_zero_final(df)
        df = clean_prediction (df)
        # Sélection de la période avec un calendrier interactif
        min_date = df["poll_date"].min()
        max_date = df["poll_date"].max()

        selected_dates = st.date_input(
            "📆 Sélectionnez une période :",
            [min_date, max_date],
            min_value=min_date,
            max_value=max_date
        )

        # Sélection du niveau de zoom avec un slider
        y_min, y_max = df[[col for col in df.columns if "prediction_result_candidate" in col]].min().min(), df[[col for col in df.columns if "prediction_result_candidate" in col]].max().max()
        zoom_level = st.slider("🔍 Zoom sur l'axe Y (%)", min_value= 0.0, max_value=float(y_max + 40), value=(0.0, float(y_max + 5)))

# Colonne 2 : Affichage du graphique
with col2:
    if os.path.exists(file_path):
        if len(selected_dates) == 2:
            start_date, end_date = pd.to_datetime(selected_dates)
            filtered_df = df[(df["poll_date"] >= start_date) & (df["poll_date"] <= end_date)]

            if filtered_df.empty:
                st.warning("⚠️ Aucune donnée disponible pour la période sélectionnée !")
            else:
                # Graphique des prédictions
                fig, ax = plt.subplots(figsize=(10, 5))

                candidate_columns = [col for col in df.columns if "prediction_result_candidate" in col]
                identity_columns = [col for col in df.columns if "identity_candidate" in col]

                for i, col in enumerate(candidate_columns):
                    candidate_name = df[identity_columns[i]].iloc[0] if identity_columns else f"Candidat {i+1}"
                    ax.plot(filtered_df["poll_date"], filtered_df[col], label=candidate_name)

                ax.set_xlabel("Date du sondage", fontsize = 12)
                ax.set_ylabel("Résultat de prédiction (%)", fontsize = 12)
                ax.set_ylim(zoom_level)  # Appliquer le zoom de l'utilisateur
                ax.grid(True, linestyle='--', alpha=0.7)
                ax.tick_params(axis='x', rotation=45)
                ax.legend()
                st.pyplot(fig)

                # Affichage des statistiques descriptives sous le graphique
                st.subheader("📊 Statistiques descriptives")
                st.write(filtered_df[candidate_columns].describe())

                # Option de téléchargement des données filtrées
                st.subheader("📥 Télécharger les données filtrées")
                csv = data.to_csv(index=False).encode('utf-8')
                st.download_button(
                    label="📂 Télécharger en CSV",
                    data=csv,
                    file_name=f"predictions_{selected_year}_{selected_country}.csv",
                    mime="text/csv"
                )
    else:
        st.error("❌ Le fichier de données sélectionné n'existe pas.")
